# DIVE-HMM · CCBF — Notebook 2 (Headline experiments)

This notebook runs the **paper's headline evaluation** for Calibrated Condition-Belief Filtering (CCBF):

1. **Multi-condition localization** — CCBF vs single-image, fixed-observation filter, SeqSLAM-style, and an
   uncertainty baseline (SUE / L2), on genuinely multi-condition data.
2. **Calibration** — ECE and **reliability diagrams** (the headline reliability metric).
3. **Open-set / off-map** — **AUROC / AUPR** for "lost" detection.
4. **Theory validation** — the **ECE-vs-sequence-length (T)** curve that directly tests *Theorem 1
   (miscalibration propagation)*: a fixed, condition-agnostic observation model should degrade with T while CCBF
   stays flat.
5. **Label-free test-time domain adaptation** — condition belief vs. true condition, and condition-drift robustness.

**Datasets.** The robust backbone is **Nordland Full** (4 synchronized seasons, clean frame-ID GT → ideal,
*controlled* multi-condition testbed for calibration/open-set; note these metrics are **not** saturated the way
recall is). **RobotCar Seasons v2** and **Extended CMU** are added as hard real-world conditions via a *best-effort,
self-probing* loader — if the pose-GT format matches the standard schema it runs; otherwise it prints the format so
the loader can be finalized in NB3.

> Run top-to-bottom. `CONFIG.SMOKE_TEST=True` (default) uses a small subset so the full suite finishes in minutes.
> Offline / Kaggle Internet OFF. Reuses the descriptor cache in `/kaggle/working/dive_hmm/features` if present.

In [ ]:
# === P0.0 — Imports, device, seeds (same contract as NB1) ==============================
import os, sys, json, time, glob, math, warnings
from dataclasses import dataclass, field, asdict
from collections import OrderedDict, defaultdict
import numpy as np

REPORT = OrderedDict()
def log(step, status, msg=""):
    REPORT[step] = {"status": status, "msg": msg}
    print({"ok": "[ OK ]", "warn": "[WARN]", "fail": "[FAIL]", "info": "[info]"}.get(status, "[??]"), step + ":", msg)

def _seed(s=0):
    import random; random.seed(s); np.random.seed(s)
    try:
        import torch; torch.manual_seed(s)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
    except Exception: pass

try:
    import torch; _HAS_TORCH = True
except Exception as e:
    _HAS_TORCH = False; print("torch not importable:", e)
_seed(0)
if _HAS_TORCH:
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    BF16_OK = (DEVICE == "cuda" and torch.cuda.is_bf16_supported())
    print("torch", torch.__version__, "| device", DEVICE, "| bf16", BF16_OK)
    if DEVICE == "cuda": print("GPU:", torch.cuda.get_device_properties(0).name)
else:
    DEVICE, BF16_OK = "cpu", False
print("numpy", np.__version__)

In [ ]:
# === P0.1 — CONFIG =====================================================================
@dataclass
class Config:
    SMOKE_TEST: bool = True
    SEED: int = 0
    INPUT_ROOTS: tuple = ("/kaggle/input",)
    WORKING_DIR: str = "/kaggle/working/dive_hmm"
    CACHE_SUBDIR: str = "features"; CKPT_SUBDIR: str = "ckpt"

    IMG_SIZE: int = 322
    BATCH_SIZE: int = 64
    NUM_WORKERS: int = 2
    USE_BF16: bool = True
    SMOKE_MAX_FRAMES: int = 800          # per season/condition in smoke

    # CCBF
    STICKY_RHO: float = 0.97
    MOTION_STD: float = 2.0
    MOTION_VELOCITY: float = 1.0
    MOTION_LEAK: float = 1e-3
    NULL_PRIOR: float = 0.02
    NULL_SELF: float = 0.9

    # evaluation
    RECALL_KS: tuple = (1, 5, 10)
    PLACE_TOL: int = 2
    ECE_BINS: int = 15
    USE_TRAIN_CONDITION_LABELS: bool = True

    # experiments
    T_GRID: tuple = (1, 2, 3, 5, 8, 13, 21, 34, 55, 89)   # sequence lengths for ECE-vs-T
    OFFMAP_MAP_FRACTION: float = 0.7      # first 70% of the route is the map; last 30% are off-map queries
    ROBOTCAR_REF_COND: str = "overcast-reference"
    ROBOTCAR_QUERY_CONDS: tuple = ("night", "night-rain", "snow", "rain", "dusk", "sun")
    POSE_TOL_M: float = 5.0               # metres: a retrieval is correct if within this of GT pose

    @property
    def cache_dir(self): return os.path.join(self.WORKING_DIR, self.CACHE_SUBDIR)
    @property
    def ckpt_dir(self): return os.path.join(self.WORKING_DIR, self.CKPT_SUBDIR)

CONFIG = Config()
if _HAS_TORCH and DEVICE == "cuda":
    g = torch.cuda.get_device_properties(0).total_memory / 1e9
    CONFIG.BATCH_SIZE = 256 if g >= 70 else (128 if g >= 40 else 64)
CONFIG.USE_BF16 = CONFIG.USE_BF16 and BF16_OK
os.makedirs(CONFIG.cache_dir, exist_ok=True); os.makedirs(CONFIG.ckpt_dir, exist_ok=True)
print(json.dumps({k: str(v) for k, v in asdict(CONFIG).items()}, indent=0)[:800])
log("P0.1.config", "ok", f"SMOKE={CONFIG.SMOKE_TEST} batch={CONFIG.BATCH_SIZE} bf16={CONFIG.USE_BF16}")

In [ ]:
# === P0.2 — Offline env + path discovery ==============================================
for k, v in {"HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "HF_DATASETS_OFFLINE": "1",
             "PIP_NO_INDEX": "1", "WANDB_MODE": "offline", "TOKENIZERS_PARALLELISM": "false"}.items():
    os.environ.setdefault(k, v)

def _find_dir(markers_any, markers_all=(), roots=None, max_depth=4):
    roots = roots or CONFIG.INPUT_ROOTS; seen = set()
    for root in roots:
        if not os.path.isdir(root): continue
        frontier = [(root, 0)]
        while frontier:
            d, dep = frontier.pop(0)
            if d in seen or dep > max_depth: continue
            seen.add(d)
            try: entries = set(os.listdir(d))
            except Exception: continue
            ok_all = all(any(m.lower() in e.lower() for e in entries) for m in markers_all) if markers_all else True
            ok_any = any(any(m.lower() in e.lower() for e in entries) for m in markers_any) if markers_any else True
            if ok_all and ok_any: return d
            for e in sorted(entries):
                sub = os.path.join(d, e)
                if os.path.isdir(sub): frontier.append((sub, dep + 1))
    return None

PATHS = {
    "bundle":        _find_dir(["checkpoints"], ["checkpoints", "sources"]),
    "nordland_full": _find_dir(["data"], ["data", "annotations"]),
    "nordland_short":_find_dir(["summer", "winter"], ["summer", "winter"]),
    "robotcar":      _find_dir(["images"], ["images", "3D-models"]),
    "cmu":           _find_dir(["slice2", "slice6", "slice7"]),
}
for n, p in PATHS.items():
    log(f"P0.2.path.{n}", "ok" if p else "warn", p or "not found")
if PATHS["bundle"]:
    for sub in ["salad", "dinov2", "Hierarchical-Localization", "LightGlue", ""]:
        p = os.path.join(PATHS["bundle"], "sources", sub) if sub else os.path.join(PATHS["bundle"], "sources")
        if os.path.isdir(p) and p not in sys.path: sys.path.insert(0, p)

In [ ]:
# === P0.3 — Pluggable feature extractor (SALAD -> DINOv2 CLS+mean -> placeholder) ======
# NOTE: DINOv2 tokens are SIGNED; we use CLS + mean pooling (the GeM-clamp bug from NB1 v1 is fixed here).
EXTRACTOR_INFO = {"name": None, "dim": None, "detail": ""}

def _try_salad():
    """serizba/salad via helper.get_* (released dino_salad.ckpt configs), bypassing VPRModel. dim=8448."""
    if not PATHS["bundle"]: return None
    ck = os.path.join(PATHS["bundle"], "checkpoints", "dino_salad.ckpt")
    salad_dir = os.path.join(PATHS["bundle"], "sources", "salad")
    if not (os.path.isfile(ck) and os.path.isdir(salad_dir)): return None
    try:
        if salad_dir not in sys.path: sys.path.insert(0, salad_dir)
        from models import helper
        backbone = helper.get_backbone("dinov2_vitb14",
            {"num_trainable_blocks": 4, "return_token": True, "norm_layer": True})
        aggregator = helper.get_aggregator("SALAD",
            {"num_channels": 768, "num_clusters": 64, "cluster_dim": 128, "token_dim": 256})
        sd = torch.load(ck, map_location="cpu"); sd = sd.get("state_dict", sd)
        bb = {k[len("backbone."):]: v for k, v in sd.items() if k.startswith("backbone.")}
        ag = {k[len("aggregator."):]: v for k, v in sd.items() if k.startswith("aggregator.")}
        backbone.load_state_dict(bb, strict=False); aggregator.load_state_dict(ag, strict=False)
        model = torch.nn.Module(); model.backbone = backbone; model.aggregator = aggregator
        model.eval().to(DEVICE)
        @torch.no_grad()
        def fwd(x): return model.aggregator(model.backbone(x))
        with torch.no_grad():
            d = fwd(torch.zeros(2, 3, CONFIG.IMG_SIZE, CONFIG.IMG_SIZE, device=DEVICE))
        if not torch.isfinite(d).all() or d.ndim != 2 or d.shape[1] < 1000: return None
        return fwd, d.shape[1]
    except Exception:
        import traceback; print("  SALAD error:"); traceback.print_exc(); return None

def _try_dinov2():
    pth = os.path.join(PATHS["bundle"], "checkpoints", "dinov2_vitb14_pretrain.pth") if PATHS["bundle"] else None
    try:
        vit_base = __import__("dinov2.models.vision_transformer", fromlist=["vit_base"]).vit_base
        vit = vit_base(patch_size=14, img_size=518, init_values=1.0, block_chunks=0)
    except Exception as e:
        print("  DINOv2 build failed:", repr(e)[:120]); return None
    try:
        if pth and os.path.isfile(pth):
            sd = torch.load(pth, map_location="cpu"); sd = sd.get("state_dict", sd)
            m, u = vit.load_state_dict(sd, strict=False); print(f"  DINOv2 weights (missing={len(m)})")
        vit.eval().to(DEVICE)
        @torch.no_grad()
        def fwd(x):
            f = vit.forward_features(x)
            if isinstance(f, dict):
                parts = []
                if f.get("x_norm_clstoken") is not None: parts.append(f["x_norm_clstoken"])
                if f.get("x_norm_patchtokens") is not None: parts.append(f["x_norm_patchtokens"].mean(1))
                return torch.cat(parts, 1) if parts else f
            return f.mean(1) if f.dim() == 3 else f
        with torch.no_grad():
            d = fwd(torch.zeros(2, 3, CONFIG.IMG_SIZE, CONFIG.IMG_SIZE, device=DEVICE))
        if not torch.isfinite(d).all(): return None
        return fwd, d.shape[1]
    except Exception as e:
        print("  DINOv2 error:", repr(e)[:120]); return None

def _placeholder():
    GRID = 4
    @torch.no_grad()
    def fwd(x):
        B, C, H, W = x.shape; gh, gw = H // GRID, W // GRID; feats = []
        for i in range(GRID):
            for j in range(GRID):
                t = x[:, :, i*gh:(i+1)*gh, j*gw:(j+1)*gw]
                feats += [t.mean((2, 3)), t.std((2, 3)), (t[:, :, :, 1:] - t[:, :, :, :-1]).abs().mean((2, 3))]
        return torch.cat(feats, 1)
    return fwd, GRID * GRID * 9

def build_extractor():
    for name, b in [("salad", _try_salad), ("dinov2", _try_dinov2)]:
        if PATHS["bundle"]:
            print("try extractor:", name); r = b()
            if r is not None:
                EXTRACTOR_INFO.update(name=name, dim=r[1], detail="bundle")
                log("P0.3.extractor", "ok", f"{name} dim={r[1]}"); return r[0]
    fwd, dim = _placeholder(); EXTRACTOR_INFO.update(name="placeholder", dim=dim, detail="DEGRADED")
    log("P0.3.extractor", "warn", f"placeholder dim={dim} (not publication-grade)"); return fwd

EXTRACT_FWD = build_extractor()

## Shared library — metrics, baselines, and the CCBF core

The place/condition filter, calibrated observation model, and metrics are the **validated core from NB1**
(same equations as `CCBF_methodology.md`). We add three evaluation tools used by the headline experiments:
a **reliability diagram**, **AUROC** for open-set detection, and the **SUE / L2** uncertainty baseline
(Zaffar et al., CVPR 2024: the L2 distance to the nearest reference is a strong training-free uncertainty).

In [ ]:
# === L.1 — Metrics =====================================================================
def cosine_sim(Q, R):
    Q = Q / (np.linalg.norm(Q, axis=1, keepdims=True) + 1e-12)
    R = R / (np.linalg.norm(R, axis=1, keepdims=True) + 1e-12)
    return Q @ R.T

def _logsumexp(x, axis=None):
    x = np.asarray(x, dtype=np.float64)
    m = np.max(x, axis=axis, keepdims=True)
    s = m + np.log(np.sum(np.exp(x - m), axis=axis, keepdims=True))
    return float(s.reshape(())) if axis is None else np.squeeze(s, axis=axis)

def recall_at_k(sim, q_ids, r_ids, ks=(1, 5, 10), tol=2):
    r_ids = np.asarray(r_ids); q_ids = np.asarray(q_ids)
    kmax = min(max(ks), sim.shape[1]); topk = np.argsort(-sim, axis=1)[:, :kmax]; out = {}
    for k in ks:
        kk = min(k, sim.shape[1])
        out[k] = float((np.abs(r_ids[topk[:, :kk]] - q_ids[:, None]) <= tol).any(1).mean())
    return out

def ece_from_conf(conf, correct, n_bins=15):
    conf = np.asarray(conf, float); correct = np.asarray(correct, float)
    bins = np.linspace(0, 1, n_bins + 1); ece, rows, N = 0.0, [], len(conf)
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        m = (conf > lo) & (conf <= hi) if i > 0 else (conf >= lo) & (conf <= hi)
        if m.sum() == 0: continue
        c, a, w = conf[m].mean(), correct[m].mean(), m.sum() / N
        ece += w * abs(a - c); rows.append((0.5 * (lo + hi), c, a, int(m.sum())))
    return float(ece), rows

def pred_correct(pred_idx, q_ids, r_ids, tol=2):
    return (np.abs(np.asarray(r_ids)[pred_idx] - np.asarray(q_ids)) <= tol).astype(float)

def fit_temperature(sim, gt_idx, grid=None):
    grid = grid if grid is not None else np.logspace(-2.2, 0.3, 30)
    v = gt_idx >= 0; s, g = sim[v], gt_idx[v]
    best, bn = grid[0], np.inf
    for tau in grid:
        lp = s / tau; lp = lp - _logsumexp(lp, axis=1)[:, None]
        nll = -lp[np.arange(len(g)), g].mean()
        if nll < bn: bn, best = nll, tau
    return float(best), float(bn)

def auroc(scores, labels):
    """Mann-Whitney AUROC; higher score => predicted positive. labels in {0,1}."""
    scores = np.asarray(scores, float); labels = np.asarray(labels, int)
    P, N = int(labels.sum()), int((labels == 0).sum())
    if P == 0 or N == 0: return float("nan")
    ranks = np.argsort(np.argsort(scores)) + 1               # ascending ranks, ties approx
    return float((ranks[labels == 1].sum() - P * (P + 1) / 2) / (P * N))

def average_precision(scores, labels):
    order = np.argsort(-np.asarray(scores)); y = np.asarray(labels, int)[order]
    tp = np.cumsum(y); prec = tp / (np.arange(len(y)) + 1); rec = tp / max(1, y.sum())
    drec = np.diff(np.concatenate([[0], rec])); return float((prec * drec).sum())

def sue_uncertainty(sim):
    """SUE/L2 (Zaffar'24): L2 distance to nearest reference = sqrt(2 - 2*max cos). Higher => less certain."""
    return np.sqrt(np.maximum(2 - 2 * sim.max(1), 0.0))

In [ ]:
# === L.2 — Motion model + fixed-observation Bayes filter (baseline) ====================
def make_motion_kernel(velocity, std):
    lo, hi = int(math.floor(velocity - 4 * std)), int(math.ceil(velocity + 4 * std))
    offs = np.arange(lo, hi + 1); w = np.exp(-0.5 * ((offs - velocity) / max(std, 1e-3)) ** 2)
    return offs, w / w.sum()

def predict_place(place, offs, w, leak):
    M = place.shape[0]; new = np.zeros_like(place)
    for off, wt in zip(offs, w):
        if off == 0: new += wt * place
        elif off > 0:
            if off < M: new[off:] += wt * place[:M - off]
        else:
            a = -off
            if a < M: new[:M - a] += wt * place[a:]
    s = new.sum()
    if s > 0: new /= s
    return (1 - leak) * new + leak / M

def run_place_filter(sim, tau, null_quantile=0.98, exit_prob=0.01):
    T, M = sim.shape; offs, w = make_motion_kernel(CONFIG.MOTION_VELOCITY, CONFIG.MOTION_STD)
    place = np.full(M, (1 - CONFIG.NULL_PRIOR) / M); null = CONFIG.NULL_PRIOR
    preds = np.empty(T, np.int64); confs = np.empty(T); nulls = np.empty(T)
    for t in range(T):
        pp = (1 - exit_prob) * predict_place(place, offs, w, CONFIG.MOTION_LEAK) + null * (1 - CONFIG.NULL_SELF) / M
        pn = CONFIG.NULL_SELF * null + exit_prob * place.sum()
        logb = np.empty(M + 1)
        logb[:M] = np.log(pp + 1e-30) + sim[t] / tau
        logb[M] = np.log(pn + 1e-30) + np.quantile(sim[t], null_quantile) / tau
        logb -= _logsumexp(logb); b = np.exp(logb)
        place, null = b[:M], b[M]; s = place.sum() + null; place, null = place / s, null / s
        preds[t], confs[t], nulls[t] = int(np.argmax(place)), float(place.max()), float(null)
    return preds, confs, nulls

In [ ]:
# === L.3 — Condition model + CCBF (validated core from NB1) ============================
def _pairwise_sq_dists(X, C):
    return np.maximum((X * X).sum(1)[:, None] + (C * C).sum(1)[None, :] - 2.0 * (X @ C.T), 0.0)

def cond_log_resp(desc, COND):
    logr = -0.5 * _pairwise_sq_dists(desc.astype(np.float32), COND["C"]) / (COND["sigma"] ** 2)
    return logr - _logsumexp(logr, axis=1)[:, None]

def _sticky_predict(bc, rho, K):
    return rho * bc + (1.0 - rho) * (1.0 - bc) / max(K - 1, 1)

def fit_condition_model(DESC, TRACKS):
    names = list(DESC.keys())
    if CONFIG.USE_TRAIN_CONDITION_LABELS and len(names) >= 2:
        C = np.stack([DESC[s].mean(0) for s in names]).astype(np.float32)
    else:
        pool = np.concatenate([DESC[s] for s in names], 0); K = max(2, len(names))
        rng = np.random.default_rng(CONFIG.SEED); C = pool[rng.choice(len(pool), K, replace=False)].copy()
        for _ in range(25):
            a = np.argmin(_pairwise_sq_dists(pool, C), 1)
            nC = np.stack([pool[a == k].mean(0) if (a == k).any() else C[k] for k in range(K)])
            if np.allclose(nC, C, atol=1e-5): C = nC; break
            C = nC
        names = [f"cond{k}" for k in range(K)]
    pool = np.concatenate([DESC[s] for s in DESC], 0)
    sigma = float(np.sqrt(np.median(_pairwise_sq_dists(pool, C).min(1))) + 1e-6)
    tau, season_list = [], list(DESC.keys())
    for i, s in enumerate(names):
        src = season_list[i] if s in DESC else season_list[i % len(season_list)]
        ref = ([o for o in season_list if o != src] or [src])[0]
        sims = cosine_sim(DESC[src], DESC[ref])
        gt = gt_match_index(TRACKS[src], TRACKS[ref]) if (src in TRACKS and ref in TRACKS) else np.arange(sims.shape[0])
        tr = (np.arange(sims.shape[0]) % 2 == 0) & (gt >= 0)
        if tr.sum() < 5: tr = (gt >= 0)
        tau.append(fit_temperature(sims[tr], gt[tr])[0])
    return dict(names=names, C=C, sigma=sigma, tau=np.asarray(tau, np.float32))

def run_ccbf(sim, query_desc, COND, null_quantile=0.98, exit_prob=0.01, return_traj=False):
    T, M = sim.shape; K = len(COND["tau"])
    inv_tau = (1.0 / np.maximum(COND["tau"], 1e-4)).astype(np.float64)
    appear = cond_log_resp(query_desc, COND); offs, w = make_motion_kernel(CONFIG.MOTION_VELOCITY, CONFIG.MOTION_STD)
    place = np.full(M, (1 - CONFIG.NULL_PRIOR) / M); null = CONFIG.NULL_PRIOR; bc = np.full(K, 1.0 / K)
    preds = np.empty(T, np.int64); confs = np.empty(T); nulls = np.empty(T)
    traj = np.empty((T, K)) if return_traj else None
    for t in range(T):
        bc = _sticky_predict(bc, CONFIG.STICKY_RHO, K)
        lbc = np.log(bc + 1e-30) + appear[t]; lbc -= _logsumexp(lbc); bc = np.exp(lbc)
        pp = (1 - exit_prob) * predict_place(place, offs, w, CONFIG.MOTION_LEAK) + null * (1 - CONFIG.NULL_SELF) / M
        pn = CONFIG.NULL_SELF * null + exit_prob * place.sum()
        A = sim[t][None, :] * inv_tau[:, None] + np.log(bc + 1e-30)[:, None]
        ll_place = _logsumexp(A, axis=0)
        ll_null = _logsumexp(np.log(bc + 1e-30) + np.quantile(sim[t], null_quantile) * inv_tau)
        logb = np.empty(M + 1); logb[:M] = np.log(pp + 1e-30) + ll_place; logb[M] = np.log(pn + 1e-30) + ll_null
        logb -= _logsumexp(logb); b = np.exp(logb)
        place, null = b[:M], b[M]; s = place.sum() + null; place, null = place / s, null / s
        preds[t], confs[t], nulls[t] = int(np.argmax(place)), float(place.max()), float(null)
        if return_traj: traj[t] = bc
    return (preds, confs, nulls, traj) if return_traj else (preds, confs, nulls)

def gt_match_index(q, r):
    pos = {fid: j for j, fid in enumerate(r.frame_ids)}
    return np.array([pos.get(fid, -1) for fid in q.frame_ids], dtype=np.int64)

log("lib", "ok", "metrics + baselines + CCBF core loaded")

In [ ]:
# === L.4 — Reliability diagram helper ==================================================
def plot_reliability(curves, title, fname=None):
    """curves: dict name -> (conf, correct). Draws reliability diagram + ECE in legend."""
    try:
        import matplotlib.pyplot as plt
    except Exception as e:
        print("(plot skipped:", repr(e)[:60], ")"); return
    fig, ax = plt.subplots(figsize=(5.2, 5))
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="perfect")
    for name, (conf, corr) in curves.items():
        ece, rows = ece_from_conf(conf, corr, CONFIG.ECE_BINS)
        if rows:
            xs = [r[1] for r in rows]; ys = [r[2] for r in rows]
            ax.plot(xs, ys, "o-", ms=4, label=f"{name} (ECE={ece:.3f})")
    ax.set_xlabel("confidence"); ax.set_ylabel("accuracy"); ax.set_title(title)
    ax.legend(fontsize=8); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    if fname:
        out = os.path.join(CONFIG.WORKING_DIR, fname); fig.tight_layout(); fig.savefig(out, dpi=110); print("saved", out)
    plt.show()

## Image extraction machinery (Track, transform, cached SALAD extraction)

In [ ]:
# === D.1 — Track, transform, cached extraction =========================================
from PIL import Image
def _frame_id(f):
    d = "".join(c for c in os.path.splitext(f)[0] if c.isdigit()); return int(d) if d else None
def _imgs(d): return [f for f in os.listdir(d) if f.lower().endswith((".png", ".jpg", ".jpeg"))] if d and os.path.isdir(d) else []

class Track:
    def __init__(self, name, condition, image_dir):
        self.name, self.condition, self.image_dir = name, condition, image_dir
        pairs = sorted([(_frame_id(f), f) for f in _imgs(image_dir) if _frame_id(f) is not None])
        self.frame_ids = [a for a, _ in pairs]; self.paths = [os.path.join(image_dir, b) for _, b in pairs]
    def __len__(self): return len(self.paths)

try:
    import torchvision.transforms as T
    def build_transform(sz): return T.Compose([T.Resize((sz, sz), antialias=True), T.ToTensor(),
                                                T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))])
    _HAS_TV = True
except Exception:
    _HAS_TV = False
    def build_transform(sz):
        mean = np.array([0.485, 0.456, 0.406], np.float32); std = np.array([0.229, 0.224, 0.225], np.float32)
        def tx(im):
            a = (np.asarray(im.resize((sz, sz)).convert("RGB"), np.float32) / 255.0 - mean) / std
            return torch.from_numpy(a.transpose(2, 0, 1).copy())
        return tx
if not _HAS_TV: CONFIG.NUM_WORKERS = 0

class ImageList(torch.utils.data.Dataset):
    def __init__(self, paths, sz): self.paths = paths; self.tx = build_transform(sz); self.sz = sz
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        try: im = Image.open(self.paths[i]).convert("RGB")
        except Exception: im = Image.new("RGB", (self.sz, self.sz))
        return self.tx(im), i

@torch.no_grad()
def extract_paths(paths, frame_ids, tag, force=False):
    cp = os.path.join(CONFIG.cache_dir, tag + ".npy"); mp = os.path.join(CONFIG.cache_dir, tag + ".json")
    if (not force) and os.path.isfile(cp) and os.path.isfile(mp):
        man = json.load(open(mp))
        if man.get("extractor") == EXTRACTOR_INFO["name"] and man.get("frame_ids") == list(frame_ids):
            print("  cache hit:", tag); return np.load(cp).astype(np.float32)
    dl = torch.utils.data.DataLoader(ImageList(paths, CONFIG.IMG_SIZE), batch_size=CONFIG.BATCH_SIZE,
                                     shuffle=False, num_workers=CONFIG.NUM_WORKERS, pin_memory=(DEVICE == "cuda"))
    feats = np.empty((len(paths), EXTRACTOR_INFO["dim"]), np.float32)
    amp = torch.bfloat16 if CONFIG.USE_BF16 else torch.float16; t0 = time.time()
    for imgs, idx in dl:
        imgs = imgs.to(DEVICE, non_blocking=True)
        if DEVICE == "cuda":
            with torch.autocast("cuda", dtype=amp): d = EXTRACT_FWD(imgs)
        else: d = EXTRACT_FWD(imgs)
        feats[idx.numpy()] = torch.nn.functional.normalize(d.float(), dim=1).cpu().numpy()
    np.save(cp, feats); json.dump({"extractor": EXTRACTOR_INFO["name"], "dim": int(EXTRACTOR_INFO["dim"]),
                                   "frame_ids": list(frame_ids)}, open(mp, "w"))
    assert np.isfinite(feats).all(), f"non-finite descriptors in {tag}"
    print(f"  extracted {tag}: {feats.shape} in {time.time()-t0:.1f}s"); return feats

## CMU loader (robust) — pick a working slice, index images by basename, pose-GT (xyz=last 3 floats)

Dates = conditions (same route, seasonal change). Resolves pose-file image names against a basename index of all
images under the slice, iterates slices until one yields >=2 usable date-conditions. Verbose diagnostics.

In [ ]:
# === CMU.0 — locate slices, parse date pose-files, resolve via basename index ========
import os, glob, numpy as np
def _slices():
    hits = []
    for r in CONFIG.INPUT_ROOTS:
        hits += glob.glob(os.path.join(r, "**", "slice*"), recursive=True)
    hits = [h for h in hits if os.path.isdir(h) and os.path.basename(h).lower().startswith("slice")]
    # sort numerically by slice number
    def num(p):
        d = "".join(c for c in os.path.basename(p) if c.isdigit()); return int(d) if d else 999
    return sorted(set(hits), key=num)
def _base(sl):
    inner = os.path.join(sl, os.path.basename(sl)); return inner if os.path.isdir(inner) else sl

MIN_PER_COND = 30; CAP = 200
def load_slice(sl, cam_filter):
    b = _base(sl); pdir = os.path.join(b, "camera-poses")
    if not os.path.isdir(pdir): return None, "no camera-poses"
    # basename index of all images under the slice base
    idx = {}
    for d, _, files in os.walk(b):
        for f in files:
            if f.lower().endswith((".jpg", ".png")): idx.setdefault(f, os.path.join(d, f))
    cond = {}; diag = []
    for pf in sorted(glob.glob(os.path.join(pdir, "*.txt"))):
        date = "".join(ch for ch in os.path.basename(pf) if ch.isdigit())[-8:]
        nlines = ncam = nres = 0; paths = []; xyz = []
        for ln in open(pf):
            t = ln.split()
            if len(t) < 8: continue
            nlines += 1
            if cam_filter and cam_filter not in t[0]: continue
            ncam += 1
            ip = idx.get(os.path.basename(t[0]))
            if ip is None: continue
            try: v = [float(x) for x in t[1:8]]
            except ValueError: continue
            nres += 1; paths.append(ip); xyz.append(v[-3:])
        diag.append((date, nlines, ncam, nres))
        if len(paths) >= MIN_PER_COND:
            if len(paths) > CAP:
                sel = np.linspace(0, len(paths)-1, CAP).round().astype(int); paths = [paths[i] for i in sel]; xyz = [xyz[i] for i in sel]
            cond["d"+date] = {"paths": paths, "xyz": np.asarray(xyz, np.float64), "fids": list(range(len(paths)))}
    return cond, diag

SLICES = _slices(); print("slices found:", [os.path.basename(s) for s in SLICES][:12], "...total", len(SLICES))
COND = {}; SLICE = None
for sl in SLICES:
    for cam in ["_c0_", "_c1_", ""]:
        c, diag = load_slice(sl, cam)
        if c and len(c) >= 2:
            COND = c; SLICE = sl
            print(f"OK slice={os.path.basename(sl)} cam='{cam}' -> {len(c)} date-conditions")
            print("  per-date (date,lines,cam,resolved):", diag[:4], "...")
            break
        elif isinstance(diag, list) and sl == SLICES[0]:
            print(f"  try slice={os.path.basename(sl)} cam='{cam}': sample diag {diag[:2]}")
    if SLICE: break
print("CMU conditions:", {k: len(v["paths"]) for k, v in COND.items()})
log("CMU.0", "ok" if len(COND) >= 2 else "fail", f"slice={os.path.basename(SLICE) if SLICE else None} conds={len(COND)}")

In [ ]:
# === CMU.1 — SALAD descriptors per date-condition ====================================
DESC = {}
for c, v in COND.items():
    DESC[c] = extract_paths(v["paths"], v["fids"], tag="cmu_" + c)
print("descriptors:", {c: d.shape for c, d in DESC.items()}, "| extractor:", EXTRACTOR_INFO["name"])
log("CMU.1", "ok" if DESC else "fail", f"extractor={EXTRACTOR_INFO['name']}")

## NB6 — Calibrated Retrieval-Score Geometry (CRSG): the contribution

NB5 showed a trivial per-frame calibrator on the top1−top2 margin fixes VPR calibration (ECE 0.47→0.006) where the
sequential filter and condition-temperature fail. Here we validate it as a real contribution under the honest
tests that matter:

- **M1 cross-condition transfer** — fit the calibrator ONLY on benign conditions, evaluate on *held-out hard*
  conditions (night/rain/snow). Deployment scenario: calibrate once on easy data, deploy under unseen appearance.
- **M2 discrimination, not just ECE** — risk–coverage curves + **AURC** and confidence-vs-correct AUROC (ECE alone
  can be gamed by a stable base rate; selective risk is the deployment metric).
- **M3 open-set** — unified reliability vs SUE vs filter-null.
- **M4 head-to-head** — single(uncal), temperature scaling, sequential filter, CCBF, **CRSG (ours)**.

CRSG = a 2–3 feature logistic calibrator on retrieval-score geometry: margin (top1−top2), NN-distance
(√(2−2·max)), score entropy → one calibrated reliability score, descriptor-agnostic, instantaneous.

In [ ]:
# === M.0 — helpers: map/episode, features, calibrators, risk-coverage ================
def pose_dist_mat(A, Bx): return np.sqrt(((A[:, None, :] - Bx[None, :, :]) ** 2).sum(-1))
def build_route_order(xyz):
    n = len(xyz)
    if n <= 2: return np.arange(n)
    start = int(np.argmax(((xyz - xyz.mean(0)) ** 2).sum(1)))
    D = pose_dist_mat(xyz, xyz); np.fill_diagonal(D, 1e18)
    order = [start]; used = np.zeros(n, bool); used[start] = True
    for _ in range(n - 1):
        d = D[order[-1]].copy(); d[used] = 1e18; nxt = int(np.argmin(d)); order.append(nxt); used[nxt] = True
    return np.array(order)
def make_map(B):
    o = build_route_order(COND[B]["xyz"]); return dict(desc=DESC[B][o], xyz=COND[B]["xyz"][o], M=len(o))
def rc_episode(Q, MAP):
    D = pose_dist_mat(COND[Q]["xyz"], MAP["xyz"]); gt = D.argmin(1); gtd = D[np.arange(len(D)), gt]
    s = np.argsort(gt, kind="stable")
    return dict(sim=cosine_sim(DESC[Q][s], MAP["desc"]), qd=DESC[Q][s], q_xyz=COND[Q]["xyz"][s], gt=gt[s], gtd=gtd[s])
def pose_correct(pred, q_xyz, map_xyz, tol):
    return (np.sqrt(((q_xyz - map_xyz[pred]) ** 2).sum(1)) <= tol).astype(float)

def geom_features(sim, k=5):
    """per-row retrieval-score geometry features (descriptor-agnostic)."""
    part = np.sort(sim, 1)[:, ::-1][:, :max(k, 2)]
    maxs = part[:, 0]; gap12 = part[:, 0] - part[:, 1]
    gap1k = part[:, 0] - part[:, 1:k].mean(1)
    p = np.exp(sim / 0.05 - _logsumexp(sim / 0.05, 1)[:, None]); ent = -(p * np.log(p + 1e-12)).sum(1)
    sue = np.sqrt(np.maximum(2 - 2 * maxs, 0.0))
    return np.stack([gap12, gap1k, maxs, sue, ent], 1), ["gap12", "gap1k", "maxsim", "sue", "entropy"]

def fit_logistic(X, y, iters=800, lr=0.3, l2=1e-4):
    """multi-feature logistic regression (standardized) -> p(correct). Returns (predict_fn)."""
    mu = X.mean(0); sd = X.std(0) + 1e-9; Z = (X - mu) / sd
    w = np.zeros(X.shape[1]); b = 0.0; n = len(Z)
    for _ in range(iters):
        p = 1 / (1 + np.exp(-(Z @ w + b)))
        gw = Z.T @ (p - y) / n + l2 * w; gb = (p - y).mean()
        w -= lr * gw; b -= lr * gb
    return lambda Xx: 1 / (1 + np.exp(-(((Xx - mu) / sd) @ w + b)))

def risk_coverage(conf, correct):
    """selective risk vs coverage. Returns (coverages, risks, AURC). Higher conf kept first."""
    order = np.argsort(-conf); err = 1.0 - correct[order]; n = len(err)
    cov = np.arange(1, n + 1) / n; risk = np.cumsum(err) / np.arange(1, n + 1)
    return cov, risk, float(risk.mean())

avail = list(DESC.keys())
REF = next((p for p in ["overcast-summer", "sun", "overcast-winter"] if p in avail), avail[0])
MAP = make_map(REF)
ALLC = [c for c in avail if c != REF]
# Protocol B (transfer stress test): fit on a MIX that contains errors, evaluate on held-out hardest.
TRANSFER_FIT = [c for c in ["dawn", "rain", "snow", "dusk"] if c in ALLC]
TRANSFER_TEST = [c for c in ["night", "night-rain"] if c in ALLC]
TOL = CONFIG.POSE_TOL_M
print("ref map =", REF, "| all query conds =", ALLC)
print("transfer: fit", TRANSFER_FIT, "-> test", TRANSFER_TEST)
RES6 = OrderedDict()
log("M.0", "ok", f"ref={REF} conds={ALLC}")

In [ ]:
# === M.1 — per-frame tables; store top-1 POSE ERROR so correctness can sweep tol =====
def build_table(conds):
    X, pe, cond_id, sim_rows, qd_rows, xyz_rows, gt_rows = [], [], [], [], [], [], []
    for ci, q in enumerate(conds):
        ep = rc_episode(q, MAP); f, FN = geom_features(ep["sim"]); pred = ep["sim"].argmax(1)
        pred_err = np.sqrt(((ep["q_xyz"] - MAP["xyz"][pred]) ** 2).sum(1))   # top-1 pose error (m)
        X.append(f); pe.append(pred_err); cond_id.append(np.full(len(pred), ci))
        sim_rows.append(ep["sim"]); qd_rows.append(ep["qd"]); xyz_rows.append(ep["q_xyz"]); gt_rows.append(ep["gt"])
    return dict(X=np.concatenate(X), pred_err=np.concatenate(pe), cond=np.concatenate(cond_id), FN=FN,
                sim=np.concatenate(sim_rows), qd=np.concatenate(qd_rows),
                xyz=np.concatenate(xyz_rows), gt=np.concatenate(gt_rows), conds=conds)

TAB = build_table(ALLC); FN = TAB["FN"]; gi = {n: i for i, n in enumerate(FN)}
TOLS = [0.5, 1.0, 2.0, 5.0]        # sweep: tight tol -> many errors -> reliability actually matters
print(f"pooled N={len(TAB['pred_err'])} | R@1 (acc) by tolerance:")
for t in TOLS:
    acc = float((TAB["pred_err"] <= t).mean())
    per = {ALLC[i]: round(float((TAB["pred_err"][TAB["cond"] == i] <= t).mean()), 2) for i in range(len(ALLC))}
    print(f"  tol={t:>4}m  acc={acc:.3f}  per-cond={per}")
print("features:", FN)
log("M.1", "ok", f"N={len(TAB['pred_err'])}")

In [ ]:
# === M.2 — pooled cross-fit calibration, swept over tolerance (reliability where errors exist) ==
def softmax_conf(sim, tau):
    lp = sim / tau; pr = np.exp(lp - _logsumexp(lp, 1)[:, None]); return pr.max(1)
rng = np.random.default_rng(CONFIG.SEED); fitm = rng.random(len(TAB["pred_err"])) < 0.5
S = TAB["sim"][fitm]; G = TAB["gt"][fitm]; TAU_G = fit_temperature(S, G)[0]

def eval_at_tol(tol):
    y = (TAB["pred_err"] <= tol).astype(float)
    Xf, yf, Xe, ye = TAB["X"][fitm], y[fitm], TAB["X"][~fitm], y[~fitm]; simE = TAB["sim"][~fitm]
    cols = [gi["gap12"], gi["sue"], gi["entropy"]]
    cals = {
        "softmax (uncal)": softmax_conf(simE, 0.05),
        "temperature":     softmax_conf(simE, TAU_G),
        "platt(margin)":   fit_logistic(Xf[:, [gi["gap12"]]], yf)(Xe[:, [gi["gap12"]]]),
        "CRSG (ours)":     fit_logistic(Xf[:, cols], yf)(Xe[:, cols]),
    }
    _, _, aurc_opt = risk_coverage(ye + 1e-6 * np.random.default_rng(0).random(len(ye)), ye)
    rows = {}
    for m, conf in cals.items():
        rows[m] = {"ECE": float(ece_from_conf(conf, ye, CONFIG.ECE_BINS)[0]),
                   "AURC": float(risk_coverage(conf, ye)[2]), "conf_auroc": float(auroc(conf, ye))}
    return rows, float(aurc_opt), float(ye.mean())

RES6["sweep"] = {}
for t in TOLS:
    rows, aopt, acc = eval_at_tol(t); RES6["sweep"][t] = {"rows": rows, "aurc_oracle": aopt, "acc": acc}
    print(f"\n[tol={t}m  acc={acc:.3f}  oracleAURC={aopt:.4f}]  {'method':<16}{'ECE':>8}{'AURC':>8}{'conf-AUROC':>11}")
    for m in rows:
        print(f"{'':>34}{m:<16}{rows[m]['ECE']:>8.4f}{rows[m]['AURC']:>8.4f}{rows[m]['conf_auroc']:>11.4f}")
# pick a meaningful-error tolerance for the headline (acc closest to 0.80)
TOL_HEAD = min(TOLS, key=lambda t: abs(RES6["sweep"][t]["acc"] - 0.80))
RES6["MA"] = RES6["sweep"][TOL_HEAD]; RES6["MA"]["tol"] = TOL_HEAD
print(f"\nheadline tolerance = {TOL_HEAD}m (acc {RES6['MA']['acc']:.3f}, enough errors for reliability to matter)")
log("M.2", "ok", f"headline tol={TOL_HEAD}m CRSG conf-AUROC={RES6['MA']['rows']['CRSG (ours)']['conf_auroc']:.3f}")

In [ ]:
# === M.3 — sequential filter & CCBF (full-sequence baselines) ========================
def build_cond_model_rc(conds, ref, MAP):
    names = list(conds); C = np.stack([DESC[c].mean(0) for c in names]).astype(np.float32)
    pool = np.concatenate([DESC[c] for c in names], 0)
    sig = float(np.sqrt(np.median(_pairwise_sq_dists(pool, C).min(1))) + 1e-6)
    tau = []
    for c in names:
        ep = rc_episode(c, MAP); k = ep["gtd"] <= TOL
        tau.append(fit_temperature(ep["sim"][k], ep["gt"][k])[0] if k.sum() >= 5 else 0.03)
    return dict(names=names, C=C, sigma=sig, tau=np.asarray(tau, np.float32))
CONDM = build_cond_model_rc(ALLC, REF, MAP)

seq = {"fixed_filter": [[], []], "ccbf": [[], []]}
for q in ALLC:
    ep = rc_episode(q, MAP)
    pf, cf, _ = run_place_filter(ep["sim"], TAU_G); seq["fixed_filter"][0].append(cf); seq["fixed_filter"][1].append(pose_correct(pf, ep["q_xyz"], MAP["xyz"], TOL_HEAD))
    pc, cc, _ = run_ccbf(ep["sim"], ep["qd"], CONDM); seq["ccbf"][0].append(cc); seq["ccbf"][1].append(pose_correct(pc, ep["q_xyz"], MAP["xyz"], TOL_HEAD))
print(f"\nsequential baselines (full sequences, all conds pooled, tol={TOL_HEAD}m):")
print(f"{'method':<18}{'ECE':>9}{'AURC':>9}{'conf-AUROC':>12}")
for name in seq:
    conf = np.concatenate(seq[name][0]); corr = np.concatenate(seq[name][1])
    ece = ece_from_conf(conf, corr, CONFIG.ECE_BINS)[0]; _, _, aurc = risk_coverage(conf, corr)
    RES6["MA"]["rows"][name] = {"ECE": float(ece), "AURC": float(aurc), "conf_auroc": float(auroc(conf, corr))}
    print(f"{name:<18}{ece:>9.4f}{aurc:>9.4f}{auroc(conf, corr):>12.4f}  (acc={corr.mean():.3f})")
log("M.3", "ok", "filter+ccbf (full-seq) compared")

In [ ]:
# === M.4 — open-set / off-map (CRSG reliability vs SUE vs filter-null) ===============
# CRSG calibrator fit on Protocol-A fit half (correctness at the headline tolerance), reused as reliability.
_yhead = (TAB["pred_err"] <= TOL_HEAD).astype(float)
CRSG_CAL = fit_logistic(TAB["X"][fitm][:, [gi["gap12"], gi["sue"], gi["entropy"]]], _yhead[fitm])
def exp_offmap(frac=0.7):
    Mmap = int(MAP["M"] * frac); msub = dict(desc=MAP["desc"][:Mmap], xyz=MAP["xyz"][:Mmap])
    ag = {m: {"s": [], "l": []} for m in ["sue", "filter_null", "crsg_unmap"]}
    for q in ALLC:
        ep = rc_episode(q, MAP); lab = (ep["gt"] >= Mmap).astype(int)
        if lab.sum() == 0 or lab.sum() == len(lab): continue
        sim2 = cosine_sim(ep["qd"], msub["desc"]); f2, _ = geom_features(sim2)
        _, _, nf = run_place_filter(sim2, TAU_G)
        crsg_rel = CRSG_CAL(f2[:, [gi["gap12"], gi["sue"], gi["entropy"]]])
        for m, sc in [("sue", f2[:, gi["sue"]]), ("filter_null", nf), ("crsg_unmap", 1 - crsg_rel)]:
            ag[m]["s"].append(sc); ag[m]["l"].append(lab)
    out = {}
    for m in ag:
        s = np.concatenate(ag[m]["s"]); l = np.concatenate(ag[m]["l"]); out[m] = {"AUROC": auroc(s, l), "AUPR": average_precision(s, l)}
    print("\noff-map detection (map=first 70% route; all conds):")
    for m in ["sue", "filter_null", "crsg_unmap"]: print(f"  {m:<14} AUROC={out[m]['AUROC']:.4f} AUPR={out[m]['AUPR']:.4f}")
    return out
RES6["M4_offmap"] = exp_offmap()
log("M.4", "ok", f"crsg={RES6['M4_offmap']['crsg_unmap']['AUROC']:.3f} sue={RES6['M4_offmap']['sue']['AUROC']:.3f}")

In [ ]:
# === M.5 — verdict ===================================================================
RH = RES6["MA"]["rows"]; th = RES6["MA"]["tol"]; off = RES6["M4_offmap"]
ours = RH["CRSG (ours)"]; aopt = RES6["MA"]["aurc_oracle"]
baselines = {m: RH[m] for m in ["temperature", "fixed_filter", "ccbf"] if m in RH}
best_base_aurc = min(b["AURC"] for b in baselines.values())
# At a meaningful-error tolerance, CRSG must (a) be well-calibrated (ECE<=0.06), (b) DISCRIMINATE
# (conf-AUROC>=0.70, i.e. not base-rate matching), (c) achieve AURC no worse than the best baseline.
crsg_cal = (ours["ECE"] <= 0.06) and (ours["conf_auroc"] >= 0.70) and (ours["AURC"] <= best_base_aurc + 0.005)
crsg_better_than_softmax = ours["ECE"] < 0.6 * RH["softmax (uncal)"]["ECE"]
crsg_offmap_ok = off["crsg_unmap"]["AUROC"] >= off["filter_null"]["AUROC"] - 0.01
summary = dict(ref=REF, allc=ALLC, headline_tol=th, headline_acc=round(RES6["MA"]["acc"], 3),
               sweep={str(t): {"acc": round(RES6["sweep"][t]["acc"], 3),
                               "ECE": {m: round(RES6["sweep"][t]["rows"][m]["ECE"], 4) for m in RES6["sweep"][t]["rows"]},
                               "conf_auroc": {m: round(RES6["sweep"][t]["rows"][m]["conf_auroc"], 4) for m in RES6["sweep"][t]["rows"]}}
                      for t in TOLS},
               headline_ece={m: round(RH[m]["ECE"], 4) for m in RH},
               headline_aurc={m: round(RH[m]["AURC"], 4) for m in RH},
               headline_conf_auroc={m: round(RH[m]["conf_auroc"], 4) for m in RH},
               aurc_oracle=round(aopt, 4), offmap={m: round(off[m]["AUROC"], 4) for m in off},
               crsg_calibrated_and_discriminative=bool(crsg_cal),
               crsg_beats_softmax=bool(crsg_better_than_softmax), crsg_offmap_ok=bool(crsg_offmap_ok))
summary["verdict"] = "CRSG_CONTRIBUTION_HOLDS" if (crsg_cal and crsg_better_than_softmax) else \
                     ("PARTIAL" if crsg_better_than_softmax else "NEEDS_WORK")
print(f"\n================ CRSG VERDICT (headline tol={th}m, acc={RES6['MA']['acc']:.3f}) ================")
print(f"CRSG      ECE={ours['ECE']:.4f} AURC={ours['AURC']:.4f} conf-AUROC={ours['conf_auroc']:.4f} (oracleAURC {aopt:.4f})")
print(f"softmax   ECE={RH['softmax (uncal)']['ECE']:.4f} AURC={RH['softmax (uncal)']['AURC']:.4f} conf-AUROC={RH['softmax (uncal)']['conf_auroc']:.4f}")
print(f"temperat  ECE={RH['temperature']['ECE']:.4f} AURC={RH['temperature']['AURC']:.4f} conf-AUROC={RH['temperature']['conf_auroc']:.4f}")
print(f"filter    ECE={RH['fixed_filter']['ECE']:.4f} AURC={RH['fixed_filter']['AURC']:.4f} conf-AUROC={RH['fixed_filter']['conf_auroc']:.4f}")
print(f"off-map   CRSG={off['crsg_unmap']['AUROC']:.3f}  SUE={off['sue']['AUROC']:.3f}  filter_null={off['filter_null']['AUROC']:.3f}")
print(f"VERDICT: {summary['verdict']}")
print("CRSG_JSON=" + json.dumps(summary))
log("M.5", "ok", f"verdict={summary['verdict']}")